# Galaxy10 DECaLS: Full Training Tutorial

**Project:** *Faking Galaxies with Diffusion: A Real-vs-Synthetic Benchmark for Morphology Classification*
Hunter Sandoval, University of New Mexico. Tutorial 6 of 6.

**Scope.** The project's whole training pipeline, end to end:

1. **train the generator**: Stable Diffusion 1.5 + LoRA + class table on the training images;
2. **generate synthetic galaxies** with it;
3. **train the classifiers of the paper's main experiment**, on real images only, on a 50/50 real/synthetic mix, and on
   synthetic images only, **several times with different seeds**, measuring the **training, validation and test
   loss and accuracy after every epoch** to show that they converge;
4. **data augmentation**: the same real-only training without the random flips and rotations, to show what
   augmentation does to the gap between training and held-out losses.

**FAST mode (on by default)** keeps the notebook to roughly 20-30 minutes on a free Colab T4: 200 LoRA steps at batch 4,
one generated galaxy per class, and classifiers trained for 5 epochs on a fixed-seed 10% stratified subset of the real
and synthetic training images (2 seeds each), evaluated on all validation and test images. The classifiers' synthetic
images come from the paper's own synthetic set (12,330 images, downloaded from the release), so the three training
conditions are the paper's, only smaller. `FAST = False` runs the paper's full settings (table below; the classifier
training sets are then drawn exactly as in the paper). The **final results cells** show the paper's full runs (their
logs) and evaluate the released classifiers, which were trained with the full settings.

## References

1. R. Rombach et al., "High-Resolution Image Synthesis with Latent Diffusion Models," CVPR 2022, arXiv:2112.10752;
   weights: https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5
2. E. J. Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models," ICLR 2022, arXiv:2106.09685.
3. J. Ho and T. Salimans, "Classifier-Free Diffusion Guidance," arXiv:2207.12598, 2022.
4. C. Lu et al., "DPM-Solver++: Fast Solver for Guided Sampling of Diffusion Probabilistic Models," arXiv:2211.01095.
5. K. He et al., "Deep Residual Learning for Image Recognition," CVPR 2016, arXiv:1512.03385.
6. C. Shorten and T. M. Khoshgoftaar, "A survey on Image Data Augmentation for Deep Learning," J. Big Data 6, 60, 2019.
7. P. Micikevicius et al., "Mixed Precision Training," ICLR 2018, arXiv:1710.03740.
8. Galaxy10 DECaLS: https://github.com/henrysky/Galaxy10
9. Project code, logs, model weights and synthetic set: https://github.com/HunterSandoval12/faking-galaxies-with-diffusion

# Setup

## Library requirements and installation

| Library | Minimum version | Tested version | Install command | Used for |
|---|---|---|---|---|
| Python | 3.10 | 3.14.7 | (Colab's default) | |
| PyTorch (`torch`) | 2.0 | 2.14.0 | `pip install "torch>=2.0"` | training |
| torchvision | 0.15 | 0.29.0 | `pip install "torchvision>=0.15"` | ResNet-18 and its ImageNet weights |
| diffusers | 0.27 | 0.40.0 | `pip install "diffusers>=0.27"` | Stable Diffusion's VAE, U-Net, schedulers |
| peft | 0.10 | 0.21.0 | `pip install "peft>=0.10"` | LoRA adapters |
| transformers | 4.30 | 5.17.0 | `pip install "transformers>=4.30"` | the CLIP text encoder (initial class table) |
| safetensors | 0.4 | 0.8.0 | `pip install "safetensors>=0.4"` | reading the released weights |
| NumPy | 1.23 | 2.5.2 | `pip install "numpy>=1.23"` | arrays |
| Pillow | 9.0 | 12.3.0 | `pip install "pillow>=9.0"` | images |
| Matplotlib | 3.6 | 3.11.2 | `pip install "matplotlib>=3.6"` | figures |

Google Colab preinstalls all of these, so the next cell normally installs nothing. It checks each library and runs
`pip` only for a missing or too-old one. On a **second run**, set `SKIP_INSTALL = True` to skip the check entirely.

**One conflict to remove.** Colab also preinstalls an old version of `torchao` (a quantization library this tutorial
does not use). Recent `peft` versions refuse to create LoRA adapters while a `torchao` older than 0.16 is installed
("ImportError: Found an incompatible version of torchao"), so the cell uninstalls such an old `torchao`
(`pip uninstall -y torchao`). This check runs even when `SKIP_INSTALL = True`.

In [ ]:
SKIP_INSTALL = False  # set to True when re-running the notebook to skip the installation check

import importlib.metadata
import re
import subprocess
import sys

REQUIREMENTS = {"torch": "2.0",
                "torchvision": "0.15",
                "diffusers": "0.27",
                "peft": "0.10",
                "transformers": "4.30",
                "safetensors": "0.4",
                "numpy": "1.23",
                "pillow": "9.0",
                "matplotlib": "3.6"}  # package -> minimum version


def version_tuple(v):
    return tuple(int(x) for x in re.findall(r"\d+", v)[:3])


def installed_version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None


if SKIP_INSTALL:
    print("SKIP_INSTALL = True: skipping the installation check.")
else:
    to_install = [f"{pkg}>={minimum}" for pkg, minimum in REQUIREMENTS.items()
                  if installed_version(pkg) is None or version_tuple(installed_version(pkg)) < version_tuple(minimum)]
    if to_install:
        print("Installing:", to_install)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
        print("Done. If a library was upgraded, restart the runtime (Runtime > Restart session) and run again.")
    else:
        print("All required libraries are already installed; nothing to install.")

# peft (LoRA) raises an ImportError while a torchao older than 0.16 is installed; this notebook does not use torchao.
TORCHAO_MIN = "0.16"
old_torchao = installed_version("torchao")
if old_torchao is not None and version_tuple(old_torchao) < version_tuple(TORCHAO_MIN):
    print(f"Removing torchao {old_torchao} (incompatible with peft; not used here)")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    import importlib
    importlib.invalidate_caches()  # let this running session see that torchao is gone

In [ ]:
# Verify the versions. The notebook stops here with a clear message if a requirement is not met.
assert sys.version_info >= (3, 10), f"Python >= 3.10 required, found {sys.version.split()[0]}"
torchao = installed_version("torchao")
assert torchao is None or version_tuple(torchao) >= version_tuple(TORCHAO_MIN), \
    f"torchao {torchao} is installed and breaks peft: run !pip uninstall -y torchao"
for pkg, minimum in REQUIREMENTS.items():
    have = installed_version(pkg)
    assert have is not None, f"{pkg} is not installed (pip install '{pkg}>={minimum}')"
    assert version_tuple(have) >= version_tuple(minimum), f"{pkg} {have} is older than the required {minimum}"
    print(f"{pkg:<13} {have:<14} (required >= {minimum})")
print("All version requirements are satisfied.")

## Hardware used for running

- **GPU required**: in Colab choose *Runtime > Change runtime type > T4 GPU* (the notebook asks for it when opened from
  GitHub). The T4 has 15 GB of memory and no bf16 hardware, so the notebook trains in fp16 there (next section).
- **Peak GPU memory (FAST mode, fp16, GPU limited to 15 GB like a T4):** LoRA training 6.2 GB, generation
  6.9 GB, classifier training 2.2 GB. Full settings (`FAST = False`): the LoRA training at batch 8 needs
  10.6 GB in fp16 (measured the same way), so it fits a T4 too.
- **Disk:** about 4 GB on the Colab machine (the 1.75 GB synthetic set, Stable Diffusion's 2 GB).
- **Tested on:** a Windows 11 PC with an NVIDIA RTX 5090 (32 GB), AMD Ryzen 7 9800X3D, 32 GB RAM, Python 3.14. Every
  code cell was run there in fp16 with GPU memory capped at 15 GB (to mimic the T4) and in the GPU's native bf16.

## Approximate execution times

| Step | Test PC (FAST, fp16) | Colab T4 (FAST, estimate) | Full settings on the RTX 5090 (the paper's runs) |
|---|---|---|---|
| Downloads (weights, 1.75 GB synthetic set, Stable Diffusion) | ~23 s | 2-4 min | - |
| Load the images | ~11 s | 1-2 min | - |
| Stage 1: generator (LoRA) training | ~55 s | ~5-8 min | 60 min for 10,000 steps at batch 8 (0.36 s/step, bf16) |
| Stage 2: generation (10 + 10 images) | ~11 s | ~1-2 min | 99.5 min for the 12,330-image synthetic set (0.48 s/image in batches of 20) |
| Stage 3: 6 classifier runs (3 conditions x 2 seeds) | ~55 s | ~8-12 min | 1.5 min per run (30 epochs on 12,330 images, bf16, incl. per-epoch validation) (x 14 runs in the paper's main curve) |
| Stage 4: augmentation off (2 runs) | ~18 s | ~3-4 min | 1.5 min per run |
| **Whole tutorial** | **~3.0 min** | **~25-30 min** | - |

The Colab column is an estimate: for these models a T4 is roughly 8-10 times slower than the RTX 5090. With the full
settings (`FAST = False`) a T4 would need about 8-10 hours for the LoRA training and about 15 min per classifier run.

## Expected results

- **Convergence**: the training loss falls every epoch. The validation and test losses can jump in the first epochs
  of a FAST run (see Stage 3), then fall steadily; FAST runs are short (50 steps), so they are still falling slowly when
  the learning rate reaches 0 at epoch 5. The validation and test curves lie on top of each other (the two held-out sets
  come from the same distribution) and the seeds agree. The full runs (final results) level off over 30 epochs.
- **The paper's result, in small**: synthetic-only training reaches much lower held-out accuracy than real-only; the
  50% mix is close to real-only. FAST mode (10% of the data, 5 epochs): test accuracy about 72% / 69% / 58% for real-only / 50% synthetic / synthetic-only in fp16 (70% / 69% / 55% in bf16).
- **Augmentation**: without flips and rotations the training loss falls faster but the held-out losses stay higher, so
  the gap widens: augmentation is what keeps the held-out losses converging together with the training loss.
- **Released classifiers (full settings)**: validation accuracy 86.7% / 84.0% / 66.2% and test accuracy 87.1% / 84.6% /
  67.2% for real-only / 50% synthetic / synthetic-only (seed 0, as in the paper).

# Mount Google Drive

Training uses the training, validation and testing images from Tutorial 1. Tutorial 1 (Dataset Tutorial) saved the `training` / `validation` / `testing` directories on your Google
Drive as `MyDrive/galaxy10_tutorial/galaxy10_dataset.zip`. This tutorial reads single images straight out of that zip
file with Python's `zipfile` module, so nothing needs to be unzipped. Mounting makes your Drive a folder of the Colab
machine:

```python
from google.colab import drive
drive.mount("/content/drive")   # asks you to authorise access the first time
```

Outside Colab, the cell uses the same local folders as Tutorial 1 (`./galaxy10_tutorial_drive` and
`./galaxy10_tutorial_work`, or the environment variables `GALAXY10_TUTORIAL_DRIVE` / `GALAXY10_TUTORIAL_WORK`).

In [ ]:
import os
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
T0 = time.time()
TIMES = {}  # execution time of each step (seconds), reported at the end

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/galaxy10_tutorial")  # persistent (Google Drive)
    WORK_ROOT = Path("/content")                                      # fast, wiped after the session
else:
    DRIVE_ROOT = Path(os.environ.get("GALAXY10_TUTORIAL_DRIVE", "galaxy10_tutorial_drive")).resolve()
    WORK_ROOT = Path(os.environ.get("GALAXY10_TUTORIAL_WORK", "galaxy10_tutorial_work")).resolve()
ZIP_PATH = DRIVE_ROOT / "galaxy10_dataset.zip"  # written by Tutorial 1
WEIGHTS_DIR = WORK_ROOT / "weights"             # downloaded model files
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
print("Running in Colab:", IN_COLAB)
print("Dataset zip from Tutorial 1:", ZIP_PATH, "(found)" if ZIP_PATH.exists() else "(NOT FOUND)")

In [ ]:
import io
import zipfile

from PIL import Image

CLASS_NAMES = ["Disturbed", "Merging", "Round Smooth", "In-between Round Smooth", "Cigar-Shaped Smooth",
               "Barred Spiral", "Unbarred Tight Spiral", "Unbarred Loose Spiral", "Edge-on without Bulge",
               "Edge-on with Bulge"]
assert ZIP_PATH.exists(), "Run Tutorial 1 (Dataset Tutorial) first: it saves galaxy10_dataset.zip on your Drive."
dataset_zip = zipfile.ZipFile(ZIP_PATH)


def zip_members(archive, folder):
    """[(member, label)] of <root>/<folder>/<label>_<class>/<file>.png in a zip, sorted by class and file name."""
    out = []
    for n in sorted(archive.namelist()):
        parts = n.split("/")
        if n.endswith(".png") and len(parts) == 4 and parts[1] == folder:
            out.append((n, int(parts[2].split("_")[0])))
    return out


def stratified(items, fraction, seed=0):
    """Fixed-seed stratified subset: the same fraction of every class (at least 1 image per class)."""
    if fraction >= 1:
        return items
    rng = np.random.default_rng(seed)
    labels = np.array([c for _, c in items])
    keep = []
    for c in range(len(CLASS_NAMES)):
        idx = np.flatnonzero(labels == c)
        keep += rng.choice(idx, max(1, round(fraction * len(idx))), replace=False).tolist()
    return [items[i] for i in sorted(keep)]


def load_images(archive, items):
    """uint8 images [N, 256, 256, 3] and int64 labels [N] as (CPU) torch tensors."""
    x = np.stack([np.asarray(Image.open(io.BytesIO(archive.read(m))).convert("RGB")) for m, _ in items])
    return torch.from_numpy(x), torch.tensor([c for _, c in items], dtype=torch.int64)

# FAST mode and the full settings

| Setting | FAST (default) | Full (paper) |
|---|---|---|
| Classifier training images per condition | 10% stratified subset (1,234; fixed seed) of the real and of the synthetic images | 12,330 |
| Classifier epochs / seeds per condition | 5 / 2 | 30 / 3 (the paper used 8 for real-only) |
| Classifier batch / learning rate | 128 / 1e-3 (1 warm-up epoch + cosine) | 128 / 1e-3 (1 warm-up epoch + cosine) |
| Validation / test images | all 2,607 / 2,609 | all 2,607 / 2,609 |
| LoRA training images | the 10% subset | all 12,330 |
| LoRA steps / batch / warm-up | 200 / 4 / 20 | 10,000 / 8 / 500 |
| Generated images | 1 per class | the paper's set: 12,330 (`diffusion/generate_synthetic.py`, 99.5 min for the 12,330-image synthetic set (0.48 s/image in batches of 20)) |
| Time on the RTX 5090 | minutes | LoRA 60 min for 10,000 steps at batch 8 (0.36 s/step, bf16); classifiers 1.5 min per run (30 epochs on 12,330 images, bf16, incl. per-epoch validation) |

The classifier stage always uses the paper's released synthetic set (the images the full generator produced), so its
three conditions are the paper's; the few images generated in Stage 2 show how that set was made.

In [ ]:
FAST = True  # False: the paper's full settings (hours on a T4)

if FAST:
    FRACTION, CLS_EPOCHS, SEEDS = 0.1, 5, [0, 1]
    LORA_STEPS, LORA_BATCH, LORA_WARMUP, LORA_EVAL_EVERY = 200, 4, 20, 25
else:
    FRACTION, CLS_EPOCHS, SEEDS = 1.0, 30, [0, 1, 2]
    LORA_STEPS, LORA_BATCH, LORA_WARMUP, LORA_EVAL_EVERY = 10000, 8, 500, 500
CONDITIONS = [(0.0, "Real only"), (0.5, "50% synthetic"), (1.0, "Synthetic only")]  # share of synthetic images
COLORS = {"Real only": "#2a78d6", "50% synthetic": "#1baf7a", "Synthetic only": "#eb6834"}
print(f"FAST = {FAST}: {FRACTION:.0%} of the training images, {CLS_EPOCHS} epochs x seeds {SEEDS}; "
      f"LoRA {LORA_STEPS} steps at batch {LORA_BATCH}")

# Downloads

The notebook trains its own models starting from the public pretrained ones (ImageNet ResNet-18, Stable Diffusion 1.5). It downloads the paper's **synthetic set** (`synthetic_set.zip`, 12,330 images generated by the project's Stable Diffusion 1.5-based generator, so the CreativeML Open RAIL-M license applies to them), the **training logs** of the full runs, and the **released weights** for the final results cells.
The files are attached to the project's GitHub Release `weights-v1`. A release file has a fixed, public address,
`https://github.com/HunterSandoval12/faking-galaxies-with-diffusion/releases/download/weights-v1/<file name>`, so plain `wget` downloads it; no GitHub account or login is needed. After each download
the cell checks the file's **SHA-256 checksum** against the value published in the repository's README and in the
release notes, so a corrupted or altered file stops the notebook:

| File | SHA-256 |
|---|---|
| `resnet18_real_only_seed0.safetensors` | `ad9a4e605ef50154a6f3e377c23b4f95fae3ca6712b9a5815b4dbfcdc66f40dd` |
| `resnet18_synthetic_only_seed0.safetensors` | `47c6b1be788acd2ed8283126337f7cd40585352900a38eaeafc2eb2c647d0afd` |
| `resnet18_mix50_seed0.safetensors` | `3049c05fd484655613337b3290adb4c8d2cb013a08c07297679cef745beb24f3` |
| `lora_unet.safetensors` | `9b866502137e2b9ac7783f84b9a8110578a51aeb60c93779e03627bb0faeb20a` |
| `class_embeddings.safetensors` | `87a7fb2720dc4a0dedb66c3417265b63d02ff7b829ea86465dfc11ee2468a2cb` |
| `lora_config.json` | `29dfa782205ee5875677d8cfb358c08c8aab8290442fb1526abcab5ba7b8baf1` |
| `synthetic_set.zip` | `b2e19570046a86827d821aaf49e7e36353217963b1240fe4e7aa650447f38948` |
| `training_logs.zip` | `f945a827dae1c68d3e534f92472f17b86ba136bfa541dccdf8b36392583a24d9` |

**How we figured this out.** GitHub serves every release file at `.../releases/download/<tag>/<file name>` and
redirects the request to its file storage; `wget` follows the redirect. `hashlib.sha256` computes the checksum in
Python (on Linux, `sha256sum <file>` gives the same value).

In [ ]:
import hashlib

RELEASE_URL = "https://github.com/HunterSandoval12/faking-galaxies-with-diffusion/releases/download/weights-v1"
FILES = {  # file name -> SHA-256 checksum
    "resnet18_real_only_seed0.safetensors": "ad9a4e605ef50154a6f3e377c23b4f95fae3ca6712b9a5815b4dbfcdc66f40dd",
    "resnet18_synthetic_only_seed0.safetensors": "47c6b1be788acd2ed8283126337f7cd40585352900a38eaeafc2eb2c647d0afd",
    "resnet18_mix50_seed0.safetensors": "3049c05fd484655613337b3290adb4c8d2cb013a08c07297679cef745beb24f3",
    "lora_unet.safetensors": "9b866502137e2b9ac7783f84b9a8110578a51aeb60c93779e03627bb0faeb20a",
    "class_embeddings.safetensors": "87a7fb2720dc4a0dedb66c3417265b63d02ff7b829ea86465dfc11ee2468a2cb",
    "lora_config.json": "29dfa782205ee5875677d8cfb358c08c8aab8290442fb1526abcab5ba7b8baf1",
    "synthetic_set.zip": "b2e19570046a86827d821aaf49e7e36353217963b1240fe4e7aa650447f38948",
    "training_logs.zip": "f945a827dae1c68d3e534f92472f17b86ba136bfa541dccdf8b36392583a24d9",
}


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(2**24):
            h.update(block)
    return h.hexdigest()


t = time.time()
for name, expected in FILES.items():
    path = WEIGHTS_DIR / name
    if not (path.exists() and sha256_of(path) == expected):  # skip files downloaded earlier
        url = f"{RELEASE_URL}/{name}"
        !wget -q -O "{path}" "{url}"
    assert path.exists() and sha256_of(path) == expected, \
        f"{name}: download failed or checksum mismatch; delete {path} and run this cell again"
    print(f"{name:<42} {path.stat().st_size / 2**20:6.2f} MiB   SHA-256 OK")
TIMES["download weights"] = time.time() - t

## GPU, precision and memory

Training runs in **mixed precision**: the heavy matrix products use 16-bit numbers, the trained weights stay in 32 bits.
There are two 16-bit formats. **bf16** has the same range as 32-bit floats and needs no extra care, but only GPUs with
compute capability 8.0 or newer (Ampere and later, e.g. the A100, L4 or the project's RTX 5090) have bf16 hardware.
Colab's free **T4** is a Turing GPU (compute capability 7.5): it has fast **fp16** tensor cores but no bf16 hardware,
so bf16 would be emulated and slow. `torch.cuda.is_bf16_supported()` can still report `True` on a T4, because the
emulation *works*; the cell therefore checks the compute capability itself.

With fp16, small gradients can underflow to zero, so fp16 training uses a **gradient scaler**
(`torch.amp.GradScaler`): the loss is multiplied by a large factor before backpropagation and the gradients are divided
by it again before the optimizer step; a step whose gradients overflow is skipped and the factor lowered. The project's
own runs used bf16 on the RTX 5090, so numbers from fp16 runs differ slightly.

`PRECISION = "auto"` picks bf16 when the GPU has it and fp16 otherwise; `"fp16"`, `"bf16"` or `"fp32"` force one.
After each training stage the notebook reports the peak GPU memory, to show that it fits the T4's 15 GB.

In [ ]:
import numpy as np
import torch

PRECISION = "auto"  # "auto" | "bf16" | "fp16" | "fp32"

assert torch.cuda.is_available(), "This tutorial needs a GPU: in Colab choose Runtime > Change runtime type > T4 GPU"
DEVICE = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
NATIVE_BF16 = props.major >= 8  # bf16 tensor cores: Ampere (8.x) and newer
AMP_DTYPE = {"auto": torch.bfloat16 if NATIVE_BF16 else torch.float16, "bf16": torch.bfloat16,
             "fp16": torch.float16, "fp32": torch.float32}[PRECISION]
USE_GRAD_SCALER = AMP_DTYPE == torch.float16  # only fp16 needs loss scaling
GPU_GIB = props.total_memory / 2**30
MEMORY = {}  # peak GPU memory per stage (GiB)


def memory_report(stage):
    """Record and print the peak GPU memory since the last report."""
    peak = torch.cuda.max_memory_allocated() / 2**30
    MEMORY[stage] = peak
    print(f"[{stage}] peak GPU memory {peak:.1f} GiB of {GPU_GIB:.1f} GiB")
    torch.cuda.reset_peak_memory_stats()


print(f"GPU: {props.name}, compute capability {props.major}.{props.minor}, {GPU_GIB:.1f} GiB")
print(f"native bf16 hardware: {NATIVE_BF16}; torch.cuda.is_bf16_supported(): {torch.cuda.is_bf16_supported()}")
print(f"training precision: {str(AMP_DTYPE).replace('torch.', '')}; gradient scaler: {USE_GRAD_SCALER}")

# Load the images

Real images come from Tutorial 1's zip; synthetic images from `synthetic_set.zip`
(`synthetic_set/<label>_<class>/<sample index>_seed<seed>.png`, in the order the paper drew from). Both training pools
are fixed-seed **stratified** subsets (the same fraction of every class, so the class imbalance is kept); validation and
testing use all images.

In [ ]:
t = time.time()
synthetic_zip = zipfile.ZipFile(WEIGHTS_DIR / "synthetic_set.zip")
synthetic_items = [(n, int(n.split("/")[1].split("_")[0])) for n in sorted(synthetic_zip.namelist()) if n.endswith(".png")]
real_x, real_y = load_images(dataset_zip, stratified(zip_members(dataset_zip, "training"), FRACTION, seed=0))
syn_x, syn_y = load_images(synthetic_zip, stratified(synthetic_items, FRACTION, seed=0))
val_x, val_y = load_images(dataset_zip, zip_members(dataset_zip, "validation"))
test_x, test_y = load_images(dataset_zip, zip_members(dataset_zip, "testing"))
TIMES["load images"] = time.time() - t
print(f"synthetic set: {len(synthetic_items)} images")
for name, y in (("real training pool", real_y), ("synthetic pool", syn_y), ("validation", val_y), ("testing", test_y)):
    print(f"{name:<19} {len(y):>6} images, per class {np.bincount(y.numpy(), minlength=10).tolist()}")
print(f"loaded in {TIMES['load images']:.0f} s")

# Stage 1: Train the generator

The project's LoRA training loop (Tutorial 5 explains it step by step): Stable Diffusion 1.5 frozen, LoRA adapters on
the attention projections and a CLIP-initialised class-embedding table trained with the diffusion (noise-prediction)
loss. Besides the noisy per-step loss the loop reports the loss on a fixed evaluation batch (one validation image per
class, fixed noise and noise levels), which moves only when the model does.

In [ ]:
import copy
import math

import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import ResNet18_Weights, resnet18

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def to_input(images_u8, augment):
    """uint8 NHWC on the GPU -> normalised float NCHW; optional random flip + 90-degree rotation (as in the project)."""
    x = images_u8.permute(0, 3, 1, 2).float().div_(255.0)
    if augment:
        flip = torch.rand(x.shape[0], device=x.device) < 0.5
        x = torch.where(flip[:, None, None, None], x.flip(-1), x)
        k = torch.randint(0, 4, (x.shape[0],), device=x.device)
        x = torch.stack([torch.rot90(xi, int(ki), dims=(-2, -1)) for xi, ki in zip(x, k)])
    return (x - IMAGENET_MEAN) / IMAGENET_STD


def build_classifier(pretrained=True):
    """torchvision ResNet-18 (ImageNet weights, downloaded on first use) with a new 10-class output layer."""
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    return model.to(DEVICE).to(memory_format=torch.channels_last)


@torch.no_grad()
def evaluate(model, x_u8, y, batch_size=256):
    """Mean cross-entropy loss and accuracy of a model on uint8 images."""
    model.eval()
    loss, correct = 0.0, 0
    for s in range(0, len(x_u8), batch_size):
        xb = to_input(x_u8[s:s + batch_size].to(DEVICE), augment=False).contiguous(memory_format=torch.channels_last)
        yb = y[s:s + batch_size].to(DEVICE)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP_DTYPE != torch.float32):
            logits = model(xb).float()
        loss += F.cross_entropy(logits, yb, reduction="sum").item()
        correct += (logits.argmax(1) == yb).sum().item()
    return loss / len(x_u8), correct / len(x_u8)


def train_classifier(train_x, train_y, eval_sets, epochs, seed, augment=True, learning_rate=1e-3,
                     weight_decay=0.05, batch_size=128, warmup_epochs=1.0, verbose=True):
    """The project's classifier training (classifier/train_classifier.py): AdamW, linear warm-up for one epoch then
    cosine decay to 0 (per step), cross-entropy, mixed precision. After every epoch it evaluates every set in
    eval_sets ({name: (images, labels)}); the returned model is the epoch with the best VALIDATION accuracy.
    Returns (model, per-epoch history, per-step training losses, selected epoch)."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = build_classifier()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    steps_per_epoch = math.ceil(len(train_x) / batch_size)
    total, warmup = epochs * steps_per_epoch, int(warmup_epochs * steps_per_epoch)

    def lr_factor(step):
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)
    history, step_losses, best_acc, best_state, best_epoch = [], [], -1.0, None, 0
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(train_x))
        loss_sum, correct = 0.0, 0
        for s in range(0, len(perm), batch_size):
            bi = perm[s:s + batch_size]
            x = to_input(train_x[bi].to(DEVICE, non_blocking=True), augment).contiguous(memory_format=torch.channels_last)
            y = train_y[bi].to(DEVICE)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP_DTYPE != torch.float32):
                logits = model(x)
            loss = F.cross_entropy(logits.float(), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            step_losses.append(loss.item())
            loss_sum += loss.item() * len(bi)
            correct += (logits.argmax(1) == y).sum().item()
        rec = {"epoch": epoch, "train_loss": loss_sum / len(train_x), "train_accuracy": correct / len(train_x),
               "lr": scheduler.get_last_lr()[0]}
        for name, (ex, ey) in eval_sets.items():
            rec[f"{name}_loss"], rec[f"{name}_accuracy"] = evaluate(model, ex, ey)
        rec["elapsed_s"] = time.time() - t0
        history.append(rec)
        if rec["validation_accuracy"] > best_acc:  # model selection on VALIDATION only
            best_acc, best_epoch = rec["validation_accuracy"], epoch
            best_state = copy.deepcopy(model.state_dict())
        if verbose:
            print(f"  epoch {epoch:>2}/{epochs}  train loss {rec['train_loss']:.3f} acc {rec['train_accuracy']:.3f}  "
                  + "  ".join(f"{n} loss {rec[f'{n}_loss']:.3f} acc {rec[f'{n}_accuracy']:.3f}" for n in eval_sets)
                  + f"  ({rec['elapsed_s']:.0f} s)")
    model.load_state_dict(best_state)
    return model, history, step_losses, best_epoch

In [ ]:
import json

from diffusers import AutoencoderKL, DDPMScheduler, DPMSolverMultistepScheduler, UNet2DConditionModel
from diffusers.optimization import get_scheduler
from peft import LoraConfig
from peft.utils import set_peft_model_state_dict
from safetensors.torch import load_file
from transformers import CLIPTextModel, CLIPTokenizer

SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
LORA_CONFIG = json.loads((WEIGHTS_DIR / "lora_config.json").read_text())  # the released run's settings


def load_base_generator():
    """Stable Diffusion 1.5's VAE and U-Net (16-bit files from Hugging Face, frozen) and its training noise schedule."""
    vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae", variant="fp16", torch_dtype=AMP_DTYPE).to(DEVICE)
    unet = UNet2DConditionModel.from_pretrained(SD_REPO, subfolder="unet", variant="fp16",
                                                torch_dtype=AMP_DTYPE).to(DEVICE)
    vae.requires_grad_(False)
    unet.requires_grad_(False)
    vae.eval()
    return vae, unet, DDPMScheduler.from_pretrained(SD_REPO, subfolder="scheduler")


def add_lora(unet, cfg=LORA_CONFIG):
    """Add LoRA adapters (rank 8 on to_q/to_k/to_v/to_out.0); their weights are kept in fp32."""
    unet.add_adapter(LoraConfig(r=cfg["lora_rank"], lora_alpha=cfg["lora_alpha"], init_lora_weights="gaussian",
                                target_modules=cfg["lora_target_modules"]))
    params = [p for p in unet.parameters() if p.requires_grad]
    for p in params:
        p.data = p.data.float()
    return params


@torch.no_grad()
def clip_class_table(cfg=LORA_CONFIG):
    """Initial class-embedding table: CLIP's encoding of one prompt per class + the empty prompt (the null class)."""
    tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
    encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder", variant="fp16",
                                            torch_dtype=torch.float32).to(DEVICE)
    tokens = tokenizer(cfg["class_prompts"] + [""], padding="max_length", max_length=tokenizer.model_max_length,
                       truncation=True, return_tensors="pt").input_ids.to(DEVICE)
    table = encoder(tokens)[0].float()
    del encoder
    torch.cuda.empty_cache()
    return table  # [11, 77, 768]


def to_vae_input(images_u8, augment, resolution=512):
    """uint8 NHWC 256 px -> float NCHW in [-1, 1] at 512 px (preprocess in diffusion/train_lora.py)."""
    x = images_u8.permute(0, 3, 1, 2).float().div_(127.5).sub_(1.0)
    if augment:
        flip = torch.rand(x.shape[0], device=x.device) < 0.5
        x = torch.where(flip[:, None, None, None], x.flip(-1), x)
        k = torch.randint(0, 4, (x.shape[0],), device=x.device)
        x = torch.stack([torch.rot90(xi, int(ki), dims=(-2, -1)) for xi, ki in zip(x, k)])
    return F.interpolate(x, size=(resolution, resolution), mode="bicubic", align_corners=False).clamp_(-1.0, 1.0)


def fixed_eval_batch(vae, noise_scheduler, x_u8, y, seed=0):
    """A fixed evaluation batch: latents of held-out images with fixed noise and fixed noise levels, so the
    diffusion loss on it changes only when the model changes (the training loss is noisy by design)."""
    g = torch.Generator(DEVICE).manual_seed(seed)
    with torch.no_grad():
        latents = vae.encode(to_vae_input(x_u8.to(DEVICE), False).to(AMP_DTYPE)).latent_dist.mean.float() \
            * vae.config.scaling_factor
    noise = torch.randn(latents.shape, generator=g, device=DEVICE)
    t = torch.linspace(50, 950, len(latents), device=DEVICE).long()
    return {"noisy": noise_scheduler.add_noise(latents, noise, t), "noise": noise, "t": t, "y": y.to(DEVICE)}


@torch.no_grad()
def eval_diffusion_loss(unet, table, batch):
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP_DTYPE != torch.float32):
        pred = unet(batch["noisy"].to(AMP_DTYPE), batch["t"],
                    encoder_hidden_states=table[batch["y"]].to(AMP_DTYPE)).sample
    return F.mse_loss(pred.float(), batch["noise"]).item()


def train_lora(unet, vae, noise_scheduler, lora_params, table, train_x, train_y, steps, batch_size, eval_batch,
               eval_every, learning_rate=1e-4, weight_decay=0.01, warmup_steps=500, cond_dropout=0.1,
               max_grad_norm=1.0, seed=42):
    """The project's LoRA training loop (diffusion/train_lora.py): AdamW on the LoRA weights + class table, constant
    learning rate after a linear warm-up, classifier-free-guidance dropout, MSE on the predicted noise, gradient
    clipping. Returns the per-step training losses and the fixed-batch evaluation losses [(step, loss)]."""
    torch.manual_seed(seed)
    table.requires_grad_(True)
    optimizer = torch.optim.AdamW([{"params": lora_params}, {"params": [table]}], lr=learning_rate,
                                  weight_decay=weight_decay)
    scheduler = get_scheduler("constant_with_warmup", optimizer=optimizer, num_warmup_steps=warmup_steps,
                              num_training_steps=steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)
    null = table.shape[0] - 1
    step_losses, eval_losses = [], [(0, eval_diffusion_loss(unet, table, eval_batch))]
    unet.train()
    perm, pos, t0 = torch.randperm(len(train_x)), 0, time.time()
    for step in range(1, steps + 1):
        if pos + batch_size > len(perm):
            perm, pos = torch.randperm(len(train_x)), 0
        bi = perm[pos:pos + batch_size]
        pos += batch_size
        with torch.no_grad():
            pixels = to_vae_input(train_x[bi].to(DEVICE), augment=True).to(AMP_DTYPE)
            latents = vae.encode(pixels).latent_dist.sample().float() * vae.config.scaling_factor
        labels = train_y[bi].to(DEVICE)
        drop = torch.rand(len(labels), device=DEVICE) < cond_dropout  # classifier-free guidance training
        labels = torch.where(drop, torch.full_like(labels, null), labels)
        noise = torch.randn_like(latents)
        t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (len(latents),), device=DEVICE)
        noisy = noise_scheduler.add_noise(latents, noise, t)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP_DTYPE != torch.float32):
            pred = unet(noisy.to(AMP_DTYPE), t, encoder_hidden_states=table[labels]).sample
        loss = F.mse_loss(pred.float(), noise)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(lora_params + [table], max_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        step_losses.append(loss.item())
        if step % eval_every == 0 or step == steps:
            unet.eval()
            eval_losses.append((step, eval_diffusion_loss(unet, table, eval_batch)))
            unet.train()
            print(f"  step {step:>5}/{steps}  training loss (mean of the last {eval_every}) "
                  f"{np.mean(step_losses[-eval_every:]):.4f}  fixed-batch loss {eval_losses[-1][1]:.4f}  "
                  f"({time.time() - t0:.0f} s)")
    unet.eval()
    table.requires_grad_(False)
    return step_losses, eval_losses


def load_released_lora(unet):
    """Put the released LoRA weights into the U-Net's adapters; returns the released class table."""
    result = set_peft_model_state_dict(unet, load_file(WEIGHTS_DIR / "lora_unet.safetensors"))
    assert not result.unexpected_keys and not [k for k in result.missing_keys if "lora" in k]
    return load_file(WEIGHTS_DIR / "class_embeddings.safetensors")["table"].to(DEVICE)


@torch.no_grad()
def generate(unet, vae, table, labels, seeds, guidance=3.0, steps=30, resolution=512, batch_size=10):
    """Class-conditional images (Tutorial 4's sampling loop, batched): classifier-free guidance with the null class,
    30 DPM-Solver++ steps, VAE decode, Lanczos down to 256 px. One noise generator per image, as in the project."""
    scheduler = DPMSolverMultistepScheduler.from_pretrained(SD_REPO, subfolder="scheduler")
    null, images = table.shape[0] - 1, []
    for s in range(0, len(labels), batch_size):
        lab = torch.as_tensor(labels[s:s + batch_size], device=DEVICE)
        latents = torch.cat([torch.randn((1, 4, resolution // 8, resolution // 8), device=DEVICE, dtype=AMP_DTYPE,
                                         generator=torch.Generator(DEVICE).manual_seed(int(sd)))
                             for sd in seeds[s:s + batch_size]])
        scheduler.set_timesteps(steps, device=DEVICE)
        latents = latents * scheduler.init_noise_sigma
        cond = torch.cat([table[torch.full_like(lab, null)], table[lab]]).to(AMP_DTYPE)
        for t in scheduler.timesteps:
            inp = scheduler.scale_model_input(torch.cat([latents] * 2), t)
            eps_null, eps_class = unet(inp, t, encoder_hidden_states=cond).sample.chunk(2)
            latents = scheduler.step(eps_null + guidance * (eps_class - eps_null), t, latents).prev_sample
        x = vae.decode(latents / vae.config.scaling_factor).sample
        x = ((x.float() / 2 + 0.5).clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1).cpu().numpy()
        images += [np.asarray(Image.fromarray(im).resize((256, 256), Image.LANCZOS)) for im in x]
    return images

In [ ]:
t = time.time()
torch.cuda.reset_peak_memory_stats()
vae, unet, noise_scheduler = load_base_generator()
lora_params = add_lora(unet)
class_table = clip_class_table()
eval_idx = [int(np.flatnonzero(val_y.numpy() == c)[0]) for c in range(10)]
eval_batch = fixed_eval_batch(vae, noise_scheduler, val_x[eval_idx], val_y[eval_idx])
lora_losses, eval_losses = train_lora(unet, vae, noise_scheduler, lora_params, class_table, real_x, real_y,
                                      steps=LORA_STEPS, batch_size=LORA_BATCH, eval_batch=eval_batch,
                                      eval_every=LORA_EVAL_EVERY, warmup_steps=LORA_WARMUP)
TIMES["stage 1: generator"] = time.time() - t
memory_report("stage 1: generator training")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
w = min(25, len(lora_losses))
axes[0].plot(np.arange(1, len(lora_losses) + 1), lora_losses, color="#c3c2b7", lw=1, label="each step")
axes[0].plot(np.arange(w, len(lora_losses) + 1), np.convolve(lora_losses, np.ones(w) / w, mode="valid"),
             color="#eb6834", lw=2, label=f"mean of {w} steps")
axes[0].set_title("training loss", loc="left", fontsize=10)
axes[0].legend(fontsize=8)
axes[1].plot([s for s, _ in eval_losses], [l for _, l in eval_losses], "o-", color="#eb6834")
axes[1].set_title("fixed evaluation batch", loc="left", fontsize=10)
for ax in axes:
    ax.set_xlabel("step")
    ax.set_ylabel("MSE loss")
    ax.grid(alpha=0.3)
fig.suptitle("Stage 1: generator (LoRA) training", x=0.01, ha="left")
plt.tight_layout()
plt.show()
print(f"fixed-batch loss {eval_losses[0][1]:.4f} -> {eval_losses[-1][1]:.4f}")

# Stage 2: Generate synthetic galaxies

One galaxy per class with the generator just trained, using the paper's seeds for sample 0 of each class
(seed = 7,000,000,000 + 1,000,000 x class). The same seeds with the **released** generator (full training) reproduce
sample 0 of every class in the paper's synthetic set, which the classifiers below are trained on; the third row shows
those images from `synthetic_set.zip`.

In [ ]:
t = time.time()
gen_classes = list(range(10))
gen_seeds = [7_000_000_000 + c * 1_000_000 for c in gen_classes]
images_here = generate(unet, vae, class_table, gen_classes, gen_seeds)
released_table = load_released_lora(unet)
images_released = generate(unet, vae, released_table, gen_classes, gen_seeds)
paper_images = [np.asarray(Image.open(io.BytesIO(synthetic_zip.read(next(
    n for n, c in synthetic_items if c == k and n.split("/")[2].startswith("00000_")))))) for k in gen_classes]
TIMES["stage 2: generation"] = time.time() - t
memory_report("stage 2: generation")
corr = [np.corrcoef(a.astype(float).ravel(), b.astype(float).ravel())[0, 1] for a, b in zip(images_released, paper_images)]
print(f"released generator vs the paper's synthetic set, same seeds: pixel correlation {np.min(corr):.4f}-{np.max(corr):.4f}")

rows = [(f"this run ({LORA_STEPS} steps)", images_here), ("released generator", images_released),
        ("paper's synthetic set", paper_images)]
fig, axes = plt.subplots(3, 10, figsize=(16, 5.6))
for r, (label, imgs) in enumerate(rows):
    for c, img in enumerate(imgs):
        axes[r, c].imshow(img)
        axes[r, c].set_xticks([])
        axes[r, c].set_yticks([])
        if r == 0:
            axes[r, c].set_title(CLASS_NAMES[c].replace(" ", "\n", 1), fontsize=7.5)
    axes[r, 0].set_ylabel(label, fontsize=8)
plt.tight_layout()
plt.show()
del unet, vae, lora_params, class_table, released_table
torch.cuda.empty_cache()

# Stage 3: Train the classifiers several times

The paper's main experiment, per condition: every class keeps its number of training images, and a share of them
(0%, 50% or 100%) is replaced by synthetic images of the same class (the rule of `classifier/train_classifier.py
--replace-fraction`, including its random draws). Each condition is trained with every seed in `SEEDS`, with the
project's training loop (Tutorial 5), and after every epoch the loss and accuracy are measured on the **validation**
and the **test** images.

The test set is only *watched* here, as the tutorial requires: the kept model of every run is still the epoch with the
best **validation** accuracy, and nothing is chosen on test data (in the paper the test set was used exactly once, for
the final evaluation).

In FAST mode an epoch is only 10 steps, so the held-out losses can **jump up in the first epochs** before they fall:
evaluation uses the batch-norm *running* statistics, which lag behind weights that change quickly while the learning
rate is at its peak (Tutorial 5). From epoch 3 on the curves fall steadily; with the full settings (97 steps per epoch)
the effect is small.

In [ ]:
def compose(fraction, seed):
    """Training set of one condition: per class, round(fraction x n) synthetic + the rest real (the paper's rule)."""
    rng = np.random.default_rng(1000 + seed)
    xs, ys = [], []
    for c in range(10):
        real_c = np.flatnonzero(real_y.numpy() == c)
        syn_c = np.flatnonzero(syn_y.numpy() == c)
        n_syn = int(round(fraction * len(real_c)))
        n_real = len(real_c) - n_syn
        pick = real_c if n_real == len(real_c) else np.sort(rng.choice(real_c, n_real, replace=False))
        xs.append(real_x[pick])
        ys += [c] * n_real
        if n_syn:
            xs.append(syn_x[syn_c[np.sort(rng.choice(len(syn_c), n_syn, replace=False))]])
            ys += [c] * n_syn
    return torch.cat(xs), torch.tensor(ys, dtype=torch.int64)


held_out = {"validation": (val_x, val_y), "testing": (test_x, test_y)}
runs = {}  # (condition, seed, augment) -> per-epoch history
t = time.time()
torch.cuda.reset_peak_memory_stats()
for fraction, name in CONDITIONS:
    for seed in SEEDS:
        x, y = compose(fraction, seed)
        print(f"{name}, seed {seed}: {len(x)} training images ({int(round(fraction * 100))}% synthetic)")
        model, history, _, kept = train_classifier(x, y, held_out, epochs=CLS_EPOCHS, seed=seed)
        runs[(name, seed, True)] = history
        print(f"  kept epoch {kept} (best validation accuracy)")
        del model
TIMES["stage 3: classifiers"] = time.time() - t
memory_report("stage 3: classifier training")

## The losses converge

For each condition: the training loss (measured during each epoch, on augmented images), and the validation and test
losses and accuracies after each epoch; lines are seed means, bands the range over seeds.

In [ ]:
def curves(keys, title):
    """2 x n grid: loss (top) and accuracy (bottom) per epoch, one column per key = (condition, augment)."""
    fig, axes = plt.subplots(2, len(keys), figsize=(4.4 * len(keys), 6.2), sharex=True, sharey="row", squeeze=False)
    for col, (name, augment, label) in enumerate(keys):
        hs = [runs[(name, s, augment)] for s in SEEDS]
        ep = np.arange(1, len(hs[0]) + 1)
        color = COLORS[name]
        for row, (metric, scale) in enumerate((("loss", 1.0), ("accuracy", 100.0))):
            ax = axes[row, col]
            for split, ls, lw in (("train", ":", 1.5), ("validation", "-", 2.0), ("testing", "--", 1.5)):
                v = scale * np.array([[h[f"{split}_{metric}"] for h in hist] for hist in hs])
                ax.plot(ep, v.mean(0), color=color, ls=ls, lw=lw, label=split if split != "train" else "training")
                if split != "train":
                    ax.fill_between(ep, v.min(0), v.max(0), color=color, alpha=0.12, lw=0)
            ax.grid(alpha=0.3)
            ax.set_xticks(ep if len(ep) <= 10 else [1, 10, 20, 30])
        axes[0, col].set_title(f"{label} ({len(SEEDS)} seeds)", loc="left", fontsize=10)
        axes[1, col].set_xlabel("epoch")
    axes[0, 0].set_ylabel("cross-entropy loss")
    axes[1, 0].set_ylabel("accuracy (%)")
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


curves([(name, True, name) for _, name in CONDITIONS],
       f"Stage 3: training, validation and test curves ({'FAST' if FAST else 'full'} settings)")

print(f"{'condition':<16}{'val loss':>18}{'test loss':>18}{'val acc %':>16}{'test acc %':>16}   last-epoch change of val / test loss")
SUMMARY = {}
for _, name in CONDITIONS:
    hs = [runs[(name, s, True)] for s in SEEDS]
    last = lambda key, scale=1.0: scale * np.array([h[-1][key] for h in hs])  # noqa: E731
    change = [np.mean([h[-1][f"{k}_loss"] - h[-2][f"{k}_loss"] for h in hs]) for k in ("validation", "testing")]
    SUMMARY[name] = {k: float(last(k).mean()) for k in ("validation_loss", "testing_loss", "validation_accuracy",
                                                        "testing_accuracy")}
    print(f"{name:<16}{last('validation_loss').mean():>10.3f} +/- {last('validation_loss').std():.3f}"
          f"{last('testing_loss').mean():>10.3f} +/- {last('testing_loss').std():.3f}"
          f"{last('validation_accuracy', 100).mean():>10.1f} +/- {last('validation_accuracy', 100).std():.1f}"
          f"{last('testing_accuracy', 100).mean():>10.1f} +/- {last('testing_accuracy', 100).std():.1f}"
          f"   {change[0]:+.3f} / {change[1]:+.3f}")

# Stage 4: Data augmentation

All runs above use the project's augmentation: every training image is randomly flipped and rotated by a multiple of
90 degrees each time it is used (galaxy morphology does not depend on orientation, so this is 8 valid versions of every
image). The cell repeats the real-only runs **without** augmentation and compares the curves. Without it the network
sees the same pixels every epoch: its training loss falls faster, but the validation and test losses stay higher, so
the gap between training and held-out losses widens. Augmentation narrows that gap, which is why the project
uses it in every run.

In [ ]:
t = time.time()
for seed in SEEDS:
    print(f"Real only without augmentation, seed {seed}")
    model, history, _, _ = train_classifier(*compose(0.0, seed), held_out, epochs=CLS_EPOCHS, seed=seed, augment=False)
    runs[("Real only", seed, False)] = history
    del model
TIMES["stage 4: augmentation"] = time.time() - t
curves([("Real only", True, "Real only, with augmentation"), ("Real only", False, "Real only, no augmentation")],
       "Stage 4: the effect of data augmentation")
for augment in (True, False):
    hs = [runs[("Real only", s, augment)] for s in SEEDS]
    gap = np.mean([h[-1]["validation_loss"] - h[-1]["train_loss"] for h in hs])
    print(f"augmentation {'on ' if augment else 'off'}: final training loss {np.mean([h[-1]['train_loss'] for h in hs]):.3f}, "
          f"validation loss {np.mean([h[-1]['validation_loss'] for h in hs]):.3f}, "
          f"test loss {np.mean([h[-1]['testing_loss'] for h in hs]):.3f}; validation - training gap {gap:.3f}")

# Final results: the full runs and the released classifiers

## The paper's full runs

The same plot for the paper's runs with the full settings (30 epochs, all 12,330 images; 8 / 3 / 3 seeds), from their
training logs. Those runs measured the training and validation curves only: the paper evaluated the test set once, for
the kept models (next cell), not after every epoch. With the full data the validation curves converge smoothly;
synthetic-only is the exception the paper is about: its validation loss rises after epoch ~5 while its accuracy
plateaus at ~64%, because the synthetic images differ systematically from real galaxies.

In [ ]:
logs = {}
with zipfile.ZipFile(WEIGHTS_DIR / "training_logs.zip") as z:
    for member in z.namelist():
        if member.startswith("training_logs/classifier/") and member.endswith(".jsonl"):
            logs[member.split("/")[-1][:-6]] = [json.loads(l) for l in z.read(member).decode().splitlines() if l.strip()]
paper = {"Real only": "A_replace0", "50% synthetic": "A_replace0.5", "Synthetic only": "A_replace1"}
fig, axes = plt.subplots(2, 3, figsize=(13.2, 6.2), sharex=True, sharey="row")
for col, (_, name) in enumerate(CONDITIONS):
    hs = [v for k, v in sorted(logs.items()) if k.startswith(paper[name] + "_seed")]
    ep = np.arange(1, 31)
    for row, (metric, scale) in enumerate((("loss", 1.0), ("accuracy", 100.0))):
        ax = axes[row, col]
        for key, ls, lw, label in ((f"train_{metric}", ":", 1.5, "training"),
                                   (f"val_{metric}", "-", 2.0, "validation")):
            v = scale * np.array([[r[key] for r in h] for h in hs])
            ax.plot(ep, v.mean(0), color=COLORS[name], ls=ls, lw=lw, label=label)
            if key.startswith("val"):
                ax.fill_between(ep, v.min(0), v.max(0), color=COLORS[name], alpha=0.12, lw=0)
        ax.grid(alpha=0.3)
    axes[0, col].set_title(f"{name}: paper, full settings ({len(hs)} seeds)", loc="left", fontsize=10)
    axes[1, col].set_xlabel("epoch")
axes[0, 0].set_ylabel("cross-entropy loss")
axes[1, 0].set_ylabel("accuracy (%)")
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## The released classifiers

The released classifiers are seed 0 of each condition, trained with the full settings. Scoring them on all validation
and test images reproduces the paper's numbers (tiny differences in the last digit can come from the GPU and the
16-bit format). This is a re-computation of the paper's one-time test evaluation, not a new selection.

In [ ]:
from safetensors.torch import load_file

PAPER_TEST_ACC = {"Real only": 0.8708, "50% synthetic": 0.8455, "Synthetic only": 0.6719}  # seed 0, one-time evaluation
files = {"Real only": "resnet18_real_only_seed0", "50% synthetic": "resnet18_mix50_seed0",
         "Synthetic only": "resnet18_synthetic_only_seed0"}
print(f"{'released classifier':<22}{'val loss':>10}{'val acc':>10}{'test loss':>11}{'test acc':>10}{'paper test acc':>16}"
      f"{'this run (' + ('FAST' if FAST else 'full') + ') test acc':>27}")
for name, fname in files.items():
    model = build_classifier(pretrained=False)
    model.load_state_dict(load_file(WEIGHTS_DIR / f"{fname}.safetensors"))
    vl, va = evaluate(model, val_x, val_y)
    tl, ta = evaluate(model, test_x, test_y)
    print(f"{name:<22}{vl:>10.3f}{va:>10.3f}{tl:>11.3f}{ta:>10.3f}{PAPER_TEST_ACC[name]:>16.4f}"
          f"{SUMMARY[name]['testing_accuracy']:>27.3f}")
    del model

# Summary

In every run the training, validation and test losses fell together, the validation and test curves agree, and the
seeds agree; the full-length runs show them levelling off. Augmentation keeps the held-out losses close to the training loss. Trained on real
images the classifier generalises best; replacing half of them with synthetic galaxies costs little, but training on
synthetic images alone leaves a large gap on real galaxies: the paper's main finding.

In [ ]:
TIMES["whole notebook"] = time.time() - T0
for step, seconds in TIMES.items():
    print(f"  {step:<26} {seconds / 60:5.1f} min")
print("peak GPU memory:", {k: f"{v:.1f} GiB" for k, v in MEMORY.items()})